In [ ]:
# Setup
from pathlib import Path
import sys, json
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np

from data import load_data, TRUE_ELASTICITY, UNIT_COSTS
from model import (estimate_elasticity, estimate_elasticity_by_tier,
                   find_optimal_price, find_optimal_profit_price,
                   revenue_function, profit_function, save_model)
from utils import save_fig, save_json, plot_demand_curve, plot_revenue_vs_price, plot_elasticity_by_tier

ARTEFACTS_DIR = PROJECT_ROOT / 'artefacts'
ARTEFACTS_DIR.mkdir(exist_ok=True)
np.random.seed(42)

In [ ]:
# 1. Load data + estimate elasticity (whole corpus)
df = load_data()
est = estimate_elasticity(df)
print(f"Estimated elasticity: {est['elasticity']:.4f}  (true: {TRUE_ELASTICITY})")
print(f"Intercept: {est['intercept']:.4f}")
print(f"R-squared: {est['r_squared']:.4f}")
print(f"Std. error: {est['std_err']:.4f}")
print(f"Sample size: {est['n']}")

In [ ]:
# 2. Per-tier elasticity
tier_df = estimate_elasticity_by_tier(df)
print(tier_df.to_string(index=False))

In [ ]:
# 3. Plot elasticity by tier
plot_elasticity_by_tier(tier_df)
save_fig('03_elasticity_by_tier.png')
plt.show()

In [ ]:
# 4. Plot demand curve
plot_demand_curve(est['intercept'], est['elasticity'], price_range=(8, 60))
save_fig('04_demand_curve.png')
plt.show()

In [ ]:
# 5. Find optimal prices per tier
optima = []
for tier, unit_cost in UNIT_COSTS.items():
    tier_est = estimate_elasticity(df[df['product_tier'] == tier])
    opt = find_optimal_price(tier_est['intercept'], tier_est['elasticity'], unit_cost,
                              min_markup=0.10, max_markup=2.0)
    profit_opt = find_optimal_profit_price(tier_est['intercept'], tier_est['elasticity'], unit_cost,
                                            min_markup=0.10, max_markup=2.0)
    optima.append({
        'tier': tier,
        'unit_cost': unit_cost,
        'elasticity': tier_est['elasticity'],
        'rev_optimal_price': opt['optimal_price'],
        'rev_optimal_revenue': opt['optimal_revenue'],
        'profit_optimal_price': profit_opt['optimal_price'],
        'profit_optimal_profit': profit_opt['optimal_profit'],
        'closed_form_price': opt['closed_form_price'],
    })
optima_df = pd.DataFrame(optima)
print(optima_df.to_string(index=False))

In [ ]:
# 6. Visualise revenue/profit curves for the standard tier
std_est = estimate_elasticity(df[df['product_tier'] == 'standard'])
std_opt = find_optimal_price(std_est['intercept'], std_est['elasticity'], UNIT_COSTS['standard'])
plot_revenue_vs_price(std_est['intercept'], std_est['elasticity'], UNIT_COSTS['standard'],
                       optimal_price=std_opt['optimal_price'],
                       price_range=(10, 60))
save_fig('05_revenue_profit_vs_price.png')
plt.show()

In [ ]:
# 7. Save results
results = {
    'overall_elasticity': est,
    'tier_elasticities': tier_df.to_dict(orient='records'),
    'optima_by_tier': optima,
    'true_elasticity': TRUE_ELASTICITY,
}
save_json(results, ARTEFACTS_DIR / 'pricing_results.json')
print(f"Saved → {ARTEFACTS_DIR / 'pricing_results.json'}")